# Star against snowflake: one school, two schemas

The same exam results stored twice, in the two shapes of an analysis database that Session 3 introduced:

- a **star schema**: one fact table `star_scores` in the middle and **flat** dimension tables around it. A student row carries the name of the programme and of its faculty as text; a course row carries the name of its department.
- a **snowflake schema**: the same fact table `snowflake_scores`, but the dimensions are split further. A student points to a programme, the programme points to a faculty, a course points to a department. Each name is stored once.

The data are those of the [SQL playground](sql-playground.ipynb): seven students, three courses, thirteen exam results. Every section runs the same task on both schemas, so you can compare the SQL, the number of rows it changes and what can go wrong.

![The star schema above, with 3 tables, and the snowflake schema below, with 6 tables. Both have the fact table scores in the middle; in the star, the student and course rows repeat the names of programme, faculty and department, while the snowflake moves each of them into a table of its own](../figures/er-star-vs-snowflake.png)

**How to read the diagrams.** Each box is a table, each row a column. `PK` marks the primary key, `FK` a foreign key that points to the primary key of another table. A line joins a foreign key to the key it points to: the two bars `||` mean *exactly one*, the crow's foot `<` means *many*. So `star_students ||──< star_scores` reads *one student has many exam results*. Orange is the fact table, blue a dimension, green a level that the snowflake splits out of a dimension, light orange a column whose text repeats in every row.

To start again from an empty database, restart the kernel and run all cells.

## 1 · Start an empty PostgreSQL server

As in the SQL playground: `pgserver.get_server(folder)` starts a private PostgreSQL server in a new temporary folder, and `cleanup_mode="delete"` deletes the folder when the kernel stops. The notebook connects to the server's default database, `postgres`; the course database `airbnb` of the lecture notebooks is not needed here. The helper `q()` runs one statement and returns its rows as a DataFrame.

In [ ]:
import tempfile

import pandas as pd
import pixeltable_pgserver as pgserver
import psycopg
from psycopg.types.numeric import FloatLoader

server = pgserver.get_server(tempfile.mkdtemp(prefix="star-snowflake-"), cleanup_mode="delete")
con = psycopg.connect(server.get_uri("postgres"), autocommit=True)   # database "postgres", the default of a new server
con.adapters.register_loader("numeric", FloatLoader)


def q(sql, params=None):
    """Run one SQL statement. A query returns its rows as a DataFrame; other statements return None."""
    with con.cursor() as cur:
        cur.execute(sql, params)
        if cur.description is None:
            return None
        return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description]).convert_dtypes()


def changed(sql, params=None):
    """Run an UPDATE, INSERT or DELETE and return how many rows it changed."""
    with con.cursor() as cur:
        cur.execute(sql, params)
        return cur.rowcount


q("SELECT current_database() AS database, version()")

## 2 · The star schema: flat dimensions

`star_students` holds the programme **and** its faculty as text in every student row; `star_courses` holds the department as text in every course row. The fact table `star_scores` has one row per exam, with a foreign key to each dimension.

```mermaid
erDiagram
    direction LR
    star_students ||--o{ star_scores  : "takes exams"
    star_scores   }o--|| star_courses : "is examined in"
    star_students {
        int  student_id PK
        text name
        text programme  "repeated in every row"
        text faculty    "repeated in every row"
        int  start_year
    }
    star_scores {
        int     student_id PK, FK
        text    course_id  PK, FK
        date    exam_date
        numeric score
    }
    star_courses {
        text course_id  PK
        text title
        int  credits
        text department "repeated in every row"
    }
    classDef fact      fill:#fde3d8,stroke:#eb6834,stroke-width:3px
    classDef dimension fill:#dbe9fa,stroke:#2a78d6,stroke-width:3px
    class star_scores fact
    class star_students,star_courses dimension
```

*Colours:* orange is the fact table, blue a dimension, green a level split out of a dimension. In Mermaid's notation `||` is *exactly one* and `o{` *zero or more* (a student may have no exam yet, like Kai).

![ER diagram of the star schema: star_students and star_courses, each joined by one line to the fact table star_scores; the columns programme, faculty and department are marked as repeated text](../figures/er-star.png)

Every question about faculties or departments needs **one join** from the fact table: the names sit right in the dimension.

In [ ]:
q("""
DROP TABLE IF EXISTS star_scores, star_students, star_courses CASCADE;

CREATE TABLE star_students (                 -- dimension: who
    student_id  INTEGER PRIMARY KEY,
    name        TEXT NOT NULL,
    programme   TEXT NOT NULL,               -- the programme's name, once per student
    faculty     TEXT NOT NULL,               -- the programme's faculty, once per student
    start_year  INTEGER
);
CREATE TABLE star_courses (                  -- dimension: what
    course_id   TEXT PRIMARY KEY,
    title       TEXT NOT NULL,
    credits     INTEGER NOT NULL,
    department  TEXT NOT NULL                -- the department's name, once per course
);
CREATE TABLE star_scores (                   -- fact: one row per exam
    student_id  INTEGER NOT NULL REFERENCES star_students (student_id),
    course_id   TEXT    NOT NULL REFERENCES star_courses (course_id),
    exam_date   DATE    NOT NULL,
    score       NUMERIC(5, 1),
    PRIMARY KEY (student_id, course_id)
);

INSERT INTO star_students VALUES
    (1, 'Amira',  'Data Science',       'Computing', 2025),
    (2, 'Ben',    'Data Science',       'Computing', 2025),
    (3, 'Chen',   'Business Analytics', 'Business',  2024),
    (4, 'Dilara', 'Business Analytics', 'Business',  2025),
    (5, 'Emil',   'Data Science',       'Computing', 2024),
    (6, 'Lena',   'Economics',          'Business',  NULL),
    (7, 'Kai',    'Economics',          'Business',  2025);

INSERT INTO star_courses VALUES
    ('SQL',  'Relational databases and SQL', 5, 'Computer Science'),
    ('STAT', 'Statistics',                    6, 'Mathematics'),
    ('ML',   'Machine learning',              6, 'Computer Science');
""")
q("SELECT * FROM star_students")

## 3 · The snowflake schema: dimensions split into levels

The same students and courses, but every repeated name moves into a table of its own: `snowflake_faculties`, `snowflake_programmes` (each with a foreign key to its faculty) and `snowflake_departments`. A student row now holds only a `programme_id`.

```mermaid
erDiagram
    direction LR
    snowflake_faculties   ||--o{ snowflake_programmes  : "has"
    snowflake_programmes  ||--o{ snowflake_students    : "enrols"
    snowflake_students    ||--o{ snowflake_scores      : "takes exams"
    snowflake_scores      }o--|| snowflake_courses     : "is examined in"
    snowflake_courses     }o--|| snowflake_departments : "is taught by"
    snowflake_faculties {
        int  faculty_id PK
        text name       "each faculty once"
    }
    snowflake_programmes {
        int  programme_id PK
        text name         "each programme once"
        int  faculty_id   FK
    }
    snowflake_students {
        int  student_id   PK
        text name
        int  programme_id FK
        int  start_year
    }
    snowflake_scores {
        int     student_id PK, FK
        text    course_id  PK, FK
        date    exam_date
        numeric score
    }
    snowflake_courses {
        text course_id     PK
        text title
        int  credits
        int  department_id FK
    }
    snowflake_departments {
        int  department_id PK
        text name          "each department once"
    }
    classDef fact      fill:#fde3d8,stroke:#eb6834,stroke-width:3px
    classDef dimension fill:#dbe9fa,stroke:#2a78d6,stroke-width:3px
    classDef split     fill:#d6f2e6,stroke:#1baf7a,stroke-width:3px
    class snowflake_scores fact
    class snowflake_students,snowflake_courses dimension
    class snowflake_faculties,snowflake_programmes,snowflake_departments split
```

![ER diagram of the snowflake schema in one row: snowflake_faculties, snowflake_programmes, snowflake_students, the fact table snowflake_scores, snowflake_courses and snowflake_departments, each joined to its neighbour by a one-to-many line](../figures/er-snowflake.png)

From the fact table to a faculty is now a **chain of three lines** (scores → students → programmes → faculties), so a query needs three joins. In return, each name is stored in exactly one row.

In [ ]:
q("""
DROP TABLE IF EXISTS snowflake_scores, snowflake_students, snowflake_courses,
                     snowflake_programmes, snowflake_faculties, snowflake_departments CASCADE;

CREATE TABLE snowflake_faculties (
    faculty_id    INTEGER PRIMARY KEY,
    name          TEXT NOT NULL UNIQUE                     -- each faculty name exists once
);
CREATE TABLE snowflake_programmes (
    programme_id  INTEGER PRIMARY KEY,
    name          TEXT NOT NULL UNIQUE,
    faculty_id    INTEGER NOT NULL REFERENCES snowflake_faculties (faculty_id)
);
CREATE TABLE snowflake_departments (
    department_id INTEGER PRIMARY KEY,
    name          TEXT NOT NULL UNIQUE
);
CREATE TABLE snowflake_students (
    student_id    INTEGER PRIMARY KEY,
    name          TEXT NOT NULL,
    programme_id  INTEGER NOT NULL REFERENCES snowflake_programmes (programme_id),
    start_year    INTEGER
);
CREATE TABLE snowflake_courses (
    course_id     TEXT PRIMARY KEY,
    title         TEXT NOT NULL,
    credits       INTEGER NOT NULL,
    department_id INTEGER NOT NULL REFERENCES snowflake_departments (department_id)
);
CREATE TABLE snowflake_scores (                            -- the same fact table as in the star
    student_id    INTEGER NOT NULL REFERENCES snowflake_students (student_id),
    course_id     TEXT    NOT NULL REFERENCES snowflake_courses (course_id),
    exam_date     DATE    NOT NULL,
    score         NUMERIC(5, 1),
    PRIMARY KEY (student_id, course_id)
);

INSERT INTO snowflake_faculties   VALUES (1, 'Computing'), (2, 'Business');
INSERT INTO snowflake_programmes  VALUES (1, 'Data Science', 1), (2, 'Business Analytics', 2), (3, 'Economics', 2);
INSERT INTO snowflake_departments VALUES (1, 'Computer Science'), (2, 'Mathematics');

INSERT INTO snowflake_students VALUES
    (1, 'Amira', 1, 2025), (2, 'Ben', 1, 2025), (3, 'Chen', 2, 2024), (4, 'Dilara', 2, 2025),
    (5, 'Emil', 1, 2024), (6, 'Lena', 3, NULL), (7, 'Kai', 3, 2025);
INSERT INTO snowflake_courses VALUES
    ('SQL', 'Relational databases and SQL', 5, 1), ('STAT', 'Statistics', 6, 2), ('ML', 'Machine learning', 6, 1);
""")
q("SELECT * FROM snowflake_students")

The fact table is the same in both schemas. One `INSERT ... SELECT` fills both with the thirteen exam results, so the two schemas hold exactly the same facts.

In [ ]:
q("""
INSERT INTO star_scores (student_id, course_id, exam_date, score) VALUES
    (1, 'SQL', '2026-01-20', 92), (1, 'STAT', '2026-02-03', 78), (1, 'ML', '2026-02-10', 85),
    (2, 'SQL', '2026-01-20', 64), (2, 'STAT', '2026-02-03', 55),
    (3, 'SQL', '2026-01-20', 88), (3, 'ML',   '2026-02-10', 91),
    (4, 'SQL', '2026-01-20', 71), (4, 'STAT', '2026-02-03', 82), (4, 'ML', '2026-02-10', NULL),
    (5, 'STAT', '2026-02-03', 49), (5, 'ML',  '2026-02-10', 67),
    (6, 'SQL', '2026-01-20', 95);

INSERT INTO snowflake_scores SELECT * FROM star_scores;   -- copy the same facts
""")
q("SELECT (SELECT COUNT(*) FROM star_scores) AS star_facts, (SELECT COUNT(*) FROM snowflake_scores) AS snowflake_facts")

## 4 · Read: the same question, two queries

**Question.** What is the average score per faculty?

In the star, the faculty sits in the student dimension: **one join** from the fact table. In the snowflake, the path is fact → student → programme → faculty: **three joins**. **Predict.** Do the two results differ?

In [ ]:
star_result = q("""
SELECT st.faculty, ROUND(AVG(s.score), 1) AS avg_score, COUNT(s.score) AS graded_exams
FROM star_scores   AS s
JOIN star_students AS st ON st.student_id = s.student_id      -- 1 join
GROUP BY st.faculty
ORDER BY st.faculty
""")
star_result

In [ ]:
snowflake_result = q("""
SELECT f.name AS faculty, ROUND(AVG(s.score), 1) AS avg_score, COUNT(s.score) AS graded_exams
FROM snowflake_scores     AS s
JOIN snowflake_students   AS st ON st.student_id  = s.student_id     -- join 1
JOIN snowflake_programmes AS p  ON p.programme_id = st.programme_id  -- join 2
JOIN snowflake_faculties  AS f  ON f.faculty_id   = p.faculty_id     -- join 3
GROUP BY f.name
ORDER BY f.name
""")
snowflake_result

In [ ]:
# the same numbers from both schemas: the shape changes the SQL, not the answer
star_result.equals(snowflake_result)

**A second question** needs both dimensions: the average score per faculty **and** department. The star needs two joins, the snowflake five. Each extra join is one more line to write and to get right, and one more table for the database to read.

In [ ]:
q("""
SELECT st.faculty, c.department, ROUND(AVG(s.score), 1) AS avg_score
FROM star_scores   AS s
JOIN star_students AS st ON st.student_id = s.student_id
JOIN star_courses  AS c  ON c.course_id   = s.course_id
GROUP BY st.faculty, c.department
ORDER BY st.faculty, c.department
""")

In [ ]:
q("""
SELECT f.name AS faculty, d.name AS department, ROUND(AVG(s.score), 1) AS avg_score
FROM snowflake_scores      AS s
JOIN snowflake_students    AS st ON st.student_id   = s.student_id
JOIN snowflake_programmes  AS p  ON p.programme_id  = st.programme_id
JOIN snowflake_faculties   AS f  ON f.faculty_id    = p.faculty_id
JOIN snowflake_courses     AS c  ON c.course_id     = s.course_id
JOIN snowflake_departments AS d  ON d.department_id = c.department_id
GROUP BY f.name, d.name
ORDER BY f.name, d.name
""")

### A view hides the joins

A **view** is a saved query with a name. It stores no data: every query on the view runs its joins again. Many snowflake warehouses offer a flat view per dimension, so that analysts read a snowflake as if it were a star.

In [ ]:
q("""
CREATE OR REPLACE VIEW snowflake_students_flat AS
SELECT st.student_id, st.name, p.name AS programme, f.name AS faculty, st.start_year
FROM snowflake_students   AS st
JOIN snowflake_programmes AS p ON p.programme_id = st.programme_id
JOIN snowflake_faculties  AS f ON f.faculty_id   = p.faculty_id
""")

# the first question again, now with one join, like the star
q("""
SELECT v.faculty, ROUND(AVG(s.score), 1) AS avg_score
FROM snowflake_scores        AS s
JOIN snowflake_students_flat AS v ON v.student_id = s.student_id
GROUP BY v.faculty
ORDER BY v.faculty
""")

## 5 · Update: rename a faculty

The faculty *Business* becomes the *School of Business*. **Predict.** How many rows does each schema have to change?

In [ ]:
star_rows = changed("UPDATE star_students SET faculty = 'School of Business' WHERE faculty = 'Business'")
snowflake_rows = changed("UPDATE snowflake_faculties SET name = 'School of Business' WHERE name = 'Business'")
print(f"star: {star_rows} rows changed, one per student of the faculty")
print(f"snowflake: {snowflake_rows} row changed, the faculty itself")

Four rows against one, for seven students. In a real university with 20,000 students the star changes thousands of rows for one new name, and every one of them must be changed, in the same transaction, or the data disagree.

## 6 · Update: what goes wrong in the star

The star stores the same fact (*which faculty a programme belongs to*, *what a programme is called*) in many rows. An update that misses some of them leaves the data in two states at once: an **update anomaly**.

**Example 1, a partial rename.** *Business Analytics* is renamed *Business Data Analytics*. The update below has one condition too many: it renames only the students who started in 2025.

In [ ]:
changed("""
UPDATE star_students SET programme = 'Business Data Analytics'
WHERE programme = 'Business Analytics' AND start_year = 2025     -- the mistake: only one cohort
""")

# one programme, two names: every report that groups by programme now splits it in two
q("SELECT programme, COUNT(*) AS students FROM star_students GROUP BY programme ORDER BY programme")

In [ ]:
# in the snowflake the name exists in one row: a rename is one row, and nothing can be left behind
changed("UPDATE snowflake_programmes SET name = 'Business Data Analytics' WHERE name = 'Business Analytics'")
q("SELECT * FROM snowflake_students_flat ORDER BY student_id")

**Example 2, a change of programme.** Ben moves from *Data Science* (faculty *Computing*) to *Economics* (*School of Business*). In the star, his row holds both the programme and the faculty. **Predict.** What happens if the update changes only the programme?

In [ ]:
changed("UPDATE star_students SET programme = 'Economics' WHERE name = 'Ben'")    # the faculty is forgotten

# Ben is now an Economics student in the faculty Computing: a combination that does not exist
q("""
SELECT programme, faculty, COUNT(*) AS students
FROM star_students
GROUP BY programme, faculty
ORDER BY programme, faculty
""")

In [ ]:
# the snowflake stores only Ben's programme; his faculty follows from it and cannot disagree
changed("UPDATE snowflake_students SET programme_id = 3 WHERE name = 'Ben'")
q("SELECT name, programme, faculty FROM snowflake_students_flat WHERE name = 'Ben'")

A check query finds such contradictions in the star: a programme must belong to exactly one faculty.

In [ ]:
q("""
SELECT programme, COUNT(DISTINCT faculty) AS faculties
FROM star_students
GROUP BY programme
HAVING COUNT(DISTINCT faculty) > 1        -- every row here is a contradiction
""")

In [ ]:
# repair the star: both columns, in every row of the programme
changed("UPDATE star_students SET programme = 'Business Data Analytics' WHERE programme = 'Business Analytics'")
changed("UPDATE star_students SET faculty = 'School of Business' WHERE programme = 'Economics'")
q("SELECT programme, faculty, COUNT(*) AS students FROM star_students GROUP BY programme, faculty ORDER BY programme")

## 7 · Insert: a programme without students

The university opens the programme *AI Engineering* in the faculty *Computing*. Nobody has enrolled yet. **Predict.** Where can each schema store it?

In [ ]:
# snowflake: the programme is a row of its own, students or not
changed("INSERT INTO snowflake_programmes VALUES (4, 'AI Engineering', 1)")

q("""
SELECT f.name AS faculty, p.name AS programme, COUNT(st.student_id) AS students
FROM snowflake_programmes   AS p
JOIN snowflake_faculties    AS f  ON f.faculty_id    = p.faculty_id
LEFT JOIN snowflake_students AS st ON st.programme_id = p.programme_id   -- LEFT JOIN keeps the empty programme
GROUP BY f.name, p.name
ORDER BY f.name, p.name
""")

The star has no table for programmes: a programme exists only as text in a student row. Without a student, *AI Engineering* cannot be stored, unless one adds an invented student (an **insert anomaly**). The same holds the other way round: delete the last student of a programme, and the programme and its faculty disappear from the data (a **delete anomaly**).

In [ ]:
# star: list the programmes. AI Engineering is missing, because no student row names it
q("SELECT DISTINCT programme, faculty FROM star_students ORDER BY programme")

## 8 · How much is stored twice

Count how often each name is written in each schema.

In [ ]:
q("""
SELECT 'star'      AS schema, 'faculty names'   AS what, COUNT(*)            AS times_written FROM star_students
UNION ALL
SELECT 'star',      'programme names', COUNT(*)            FROM star_students
UNION ALL
SELECT 'snowflake', 'faculty names',   COUNT(*)            FROM snowflake_faculties
UNION ALL
SELECT 'snowflake', 'programme names', COUNT(*)            FROM snowflake_programmes
""")

Seven against two for the faculties. With 20,000 students the star writes the faculty name 20,000 times. Text is small, and analysis databases compress repeated values well, so the storage cost rarely decides. The real cost is the one of sections 5 to 7: every copy must stay correct.

## 9 · Which one, when

| | Star schema | Snowflake schema |
|---|---|---|
| **Reading** | ✅ one join per dimension; short queries that analysts and BI tools write easily | ❌ a chain of joins per dimension; longer queries (a view can hide them) |
| **Speed of reading** | ✅ fewer joins, usually faster on large fact tables | ❌ more joins; small dimension tables keep the cost low |
| **Updating a name** | ❌ many rows; a missed row leaves two versions (update anomaly) | ✅ one row; every fact sees the change at once |
| **Consistency** | ❌ a programme can end up in two faculties; only a check query finds it | ✅ foreign keys make a wrong combination impossible |
| **Entries without facts** | ❌ a programme with no students cannot be stored | ✅ every level has its own table |
| **Storage** | ❌ repeated text in every dimension row | ✅ each name once |

**In practice.** Data warehouses for reporting usually choose the star: the data arrive in batches from a pipeline that rebuilds or updates the dimensions in one go, and analysts read far more often than anyone writes ([Kimball Group, Star schemas](https://www.kimballgroup.com/data-warehouse-business-intelligence-resources/kimball-techniques/dimensional-modeling-techniques/star-schema-olap-cube/)). A snowflake fits when a dimension is large and its upper levels change on their own (a product catalogue with categories and suppliers), or when the same levels are shared by several dimensions. Operational databases, where single rows change all day, are normalised even further than a snowflake, for exactly the anomalies of sections 6 and 7.

## 10 · Your turn

1. The department *Mathematics* becomes *Mathematics and Statistics*. Write the update for both schemas and compare the number of rows changed.
2. Add a column `dean` to the faculty. Where does it go in each schema, and how many values do you have to write?
3. Write a check query for the star that finds a course whose department is spelled in two ways.
4. Turn `star_students` into a snowflake yourself: create a table of programmes from `SELECT DISTINCT programme, faculty FROM star_students`.

## 11 · Stop the server

`server.cleanup()` stops PostgreSQL and deletes the temporary data folder. Restarting the kernel does the same.

In [ ]:
con.close()
server.cleanup()